[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haricalzi/PhD-Lectures/blob/main/DataVisualization/AI-Driven_Cardiac_Prevention_EN.ipynb)

# Data Visualization Lab: Cardiac Prevention with Artificial Intelligence

**Hari Calzi** · [hari.calzi@unimi.it](mailto:hari.calzi@unimi.it)

Welcome to this lab! Today we will step into the shoes of a **data scientist** working in the medical field.

We have the clinical records of **299 patients** who suffered from heart failure.

### Goal
We want to find out whether the data can help us group patients according to the severity of their condition, and extract potentially useful information from the patterns we find.

---

Let's get started!

In [ ]:
# install the required libraries
!pip install ucimlrepo matplotlib scikit-learn seaborn numpy pandas

In [ ]:
from ucimlrepo import fetch_ucirepo

# Message to confirm that the download has started
print("Downloading the dataset...")

# Download the original dataset using its unique ID (519)
heart_failure_clinical_records = fetch_ucirepo(id=519)

print("Dataset downloaded successfully!")

In [ ]:
# Organise the data into tables called DataFrames
# X contains the clinical features (age, blood pressure, etc.)
X = heart_failure_clinical_records.data.features

# y contains the survival label (the outcome we would like to predict)
y = heart_failure_clinical_records.data.targets

print("Data extracted and ready for analysis.")

In [ ]:
# Show general information about the dataset to understand where it comes from
print("--- General Information (Metadata) ---")
print(heart_failure_clinical_records.metadata) 

In [ ]:
# Show the description of each column
print("\n--- Variable Details ---")
print(heart_failure_clinical_records.variables)

In [ ]:
# Show the first 5 rows of table X to see the numerical values
print("Preview of the data for the first 5 patients:")
X.head()

### Why automate the search?

Above we only printed the first 5 rows, but the full table has 299 rows and 12 columns: thousands of numbers. For a human being it is almost impossible to "extract" useful information or understand which patients are similar just by scrolling through these rows.

It is like looking for patterns in a sea of digits: the human eye gets tired and confused. This is why we will use **automated techniques** that let the computer analyse thousands of data points in a few moments and find hidden connections that we would never be able to see.

---

In [ ]:
# HISTOGRAM OF PATIENT AGE

# Import the libraries used to draw the plots
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Temporarily merge the features (X) and the outcome (y) into a single table
# This will make it much easier to create coloured plots!
df = pd.concat([X, y], axis=1)

# Set a clean style for the plots (removes grey backgrounds and unnecessary lines)
sns.set_theme(style="whitegrid")

# Create the figure (the "sheet" we draw on)
plt.figure(figsize=(10, 6))

# Draw the age histogram
sns.histplot(data=df, x='age', bins=15, color='#4C72B0', kde=True, alpha=0.6)

# Add a title and axis labels to make the plot clearer
plt.title('Age Distribution of Patients', fontsize=16, pad=15)
plt.xlabel('Age (years)', fontsize=14)
plt.ylabel('Number of Patients', fontsize=14)

# Show the plot
plt.show()

In [ ]:
# BAR CHART OF PATIENTS' CLINICAL OUTCOME

plt.figure(figsize=(8, 6))
outcome_palette = {'Survived': '#55A868', 'Deceased': '#C44E52'}

# In the dataset, 0 means Survived and 1 means Deceased.
df_plot = df.copy()
df_plot['death_event_label'] = df_plot['death_event'].map({0: 'Survived', 1: 'Deceased'})

# Create the bar chart with clearly distinct colours
ax = sns.countplot(
    data=df_plot,
    x='death_event_label',
    palette=outcome_palette,
    hue='death_event_label',
    alpha=0.6
    )

plt.title('Clinical Outcome of Patients', fontsize=16, pad=15)
plt.xlabel('', fontsize=14) # Left empty, the labels already say it all
plt.ylabel('Number of Patients', fontsize=14)

# Add labels above each bar to show the exact numbers
for container in ax.containers:
    ax.bar_label(container, fontsize=12, padding=3)

plt.show()

In [ ]:
# BOXPLOT OF EJECTION FRACTION FOR SURVIVED AND DECEASED PATIENTS

# Ejection fraction is a measure of heart health: the higher, the better (the heart pumps more blood with each beat)

plt.figure(figsize=(9, 6))
sns.boxplot(
    data=df_plot,
    x='death_event_label',
    y='ejection_fraction',
    hue='death_event_label',
    palette=outcome_palette,
    legend=False,
    width=0.5,
    notch=True,
    boxprops={'alpha': 0.7}
)

plt.title('Heart Efficiency: Survived vs Deceased', fontsize=16, pad=15)
plt.xlabel('', fontsize=14)
plt.ylabel('Ejection Fraction (%)', fontsize=14)

plt.show()

### How to read a boxplot

The **boxplot** is one of the most powerful tools to see how data are distributed. Here is how to interpret each of its parts:

1. **The box:** represents the core of the data, i.e. the **central 50%** of the sample.
   * The lower edge is the **25th percentile** (Q1): 25% of patients have a value below it.
   * The upper edge is the **75th percentile** (Q3): 75% of patients have a value below it.
2. **The central line:** represents the **median** (50th percentile). It is the value that splits the patients exactly in half. If the line is not in the middle of the box, the data are skewed (asymmetric).
3. **The whiskers:** the lines extending above and below the box. They show the range where the "normal" data lie (usually up to 1.5 times the height of the box).
4. **The dots (outliers):** if you see points beyond the whiskers, these are the so-called outliers. They are patients with very rare or anomalous values compared to the rest of the group.
5. **Notch:** helps to compare medians: if the notches of two groups do not overlap, the difference between the medians of the two groups is very likely to be statistically significant.

In [ ]:
# SCATTER PLOT: RELATIONSHIP BETWEEN HEART AND KIDNEY HEALTH

# Ejection fraction is a measure of heart health: the higher, the better (the heart pumps more blood with each beat)

# Serum creatinine is a measure of kidney health: the higher, the worse (the kidneys cannot filter out toxins)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=df_plot,
    x='ejection_fraction',
    y='serum_creatinine',
    hue='death_event_label',
    palette=outcome_palette,
    s=80,
    alpha=0.7
)

plt.title('Heart vs Kidneys: Relationship with Clinical Outcome', fontsize=16, pad=15)
plt.xlabel('Ejection Fraction (%, Heart Efficiency)', fontsize=14)
plt.ylabel('Serum Creatinine (mg/dL, Kidney Toxins)', fontsize=14)

plt.legend(title='Outcome', fontsize=12, title_fontsize=12)

plt.show()

---

### What is clustering?

**Clustering** is an **unsupervised learning** technique. In this type of analysis we do not give the computer the "right answers" (such as already knowing who is sick and who is not); instead, we ask it to look at the data and group together the patients that are most similar to each other.



**A practical example:** imagine having to organise a library of thousands of songs without knowing the music genres. The computer will analyse the rhythm, the instruments and the tempo to create groups (clusters) of similar tracks. In the end you might find yourself with a "rock" group and a "jazz" group, even though you never explained to it what they are!

---

### How do we choose the number of groups? ($k$)

Some algorithms, such as **K-Means**, require the researcher to decide in advance how many groups (called **$k$**) to split the data into. Picking a random number is not good practice, so we use two mathematical techniques to find the optimal value:

1. **Elbow Method:**
   * **Technical description:** it is based on the computation of the **Inertia** (or WCSS), i.e. the sum of the squared distances between each point and the centre of its group.
   * **What we look for:** the lower the inertia, the more "compact" the groups. In the plot, we look for the point where the curve "bends" sharply (like an elbow): after that point, adding more groups reduces the inertia only slightly, making the model needlessly complicated.

2. **Silhouette Method:**
   * **Technical description:** this method measures how similar a point is to its own group compared to the other groups.
   * **What we look for:** a value close to **1** means that the groups are very dense and well separated from each other. A value close to **0** indicates that the groups overlap, while negative values indicate that points have been assigned to the wrong group.

### In short: what is the difference?

To understand this better, we can imagine clusters as groups of friends in a room:

* **Elbow (Inertia) -> Looks "inside":** it measures how close each person is to the centre of their own group. It only focuses on **Cohesion**. If the group is very tight and everyone is close to the centre, the inertia is low.


* **Silhouette -> Looks "inside and outside":** it is a stricter test. It does not only ask "how close are you to your friends?", but also "how far are you from the other groups?". It therefore measures both **Cohesion** (inside) and **Separation** (outside).


**The verdict:** a "good" group according to the Silhouette is one whose members like each other (close) but, at the same time, cannot stand the other groups (far away).

In [ ]:
# Elbow Method to find the optimal number of groups (clusters)
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# WCSS (Within Cluster Sum of Squares)
# It represents the inertia: we look for the value of k where the inertia stops dropping quickly and the curve forms a sort of "elbow"
wcss = []

# Try splitting the data into a number of groups ranging from 1 to 10
k_range = range(1, 11)

# Compute the inertia for each possible number of groups
for k in k_range:
    # Initialise the K-Means algorithm
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto', max_iter=300)
    # Let the computer analyse the data
    kmeans.fit(X)
    # Store the inertia value in our list
    wcss.append(kmeans.inertia_)

# Create the "Elbow" plot
plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss, marker='o', color='blue', linestyle='--')
plt.title('Elbow Method')
plt.xlabel('Number of groups (k)')
plt.ylabel('Inertia (WCSS)')
plt.xticks(k_range)
plt.grid(True)
plt.show()


In [ ]:
# Silhouette Score analysis to validate the ideal number of groups
from sklearn.metrics import silhouette_score

# Create a list to store the score of each attempt
silhouette_scores = []

# Try different values of k (number of groups)
# Note: we start from 2 because the silhouette cannot be computed for a single group!
k_range = range(2, 11)

# Compute the score for each k
for k in k_range:
    # Initialise the K-Means algorithm
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto', max_iter=300)

    # Assign each patient to a group (labels)
    cluster_labels = kmeans.fit_predict(X)

    # Compute how "good" this split is
    # The higher the score, the better defined the groups
    score = silhouette_score(X, cluster_labels)

    # Store the result
    silhouette_scores.append(score)
    #print(f'For k = {k}, the silhouette score is {score}')

# Create the Silhouette plot
plt.figure(figsize=(10, 6))
plt.plot(k_range, silhouette_scores, marker='o', color='green', linestyle='--')
plt.title('Silhouette Score to find the optimal k')
plt.xlabel('Number of groups (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.grid(True)
plt.show()

### How does the K-Means algorithm work?

**K-Means** is one of the most popular clustering algorithms. Its goal is to split the data into $k$ groups (a number we chose ourselves thanks to the previous plots).

But how does it "physically" build these groups? It follows an iterative (repeated) process in 3 steps:

1.  **Initialisation:** the algorithm randomly places $k$ special points called **Centroids** (the "centres" of the future groups) in the plot.
2.  **Assignment:** each patient is assigned to the nearest centroid. To measure "closeness", the algorithm uses the **Euclidean Distance** (the one computed with the Pythagorean theorem).
3.  **Update:** the algorithm moves each centroid exactly to the "mathematical centre" (the mean) of all the patients assigned to it.

**The cycle repeats:** the algorithm reassigns the patients to the new centres and moves the centres again, until the groups become stable and stop changing.

![K-Means animation](https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif)

### The Euclidean Distance
For two points in the plane $P_1(x_1, y_1)$ and $P_2(x_2, y_2)$, the distance is:
$$d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$
The smaller this distance, the more similar the patient is to the "typical patient" (the centroid) of that group.

---

### How can we see data in 12 dimensions?

Our dataset has 12 features (age, blood pressure, etc.). We humans can only visualise 2 or 3 dimensions graphically (length, width and height). How can we see the groups if the data live in a 12-dimensional space?

We use **PCA (Principal Component Analysis)**, a technique that "squashes" the 12 dimensions into just 2 artificial dimensions (called components), while trying to lose as little information as possible.

---

In [ ]:
# Apply k-means with k=2 and plot the clusters after reducing to 2D with PCA

from sklearn.decomposition import PCA

# Reduce to 2 components so that we can plot the data
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X.values)

# Run K-Means on the original data (X)
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto', max_iter=300)
y_kmeans = kmeans.fit_predict(X)

# Create the cluster plot
plt.figure(figsize=(10, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_kmeans, s=50, cmap='RdYlBu', alpha=0.6)

# Compute and project the group centres
centers = kmeans.cluster_centers_
centers_pca = pca.transform(centers)

# Draw the centroids as red 'X' markers
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', s=200, alpha=0.75, marker='X')

plt.title('K-Means Clustering with k=2')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()

### Problem

The groups look strangely cut and squashed.

In [ ]:
# Plot the range of values of each feature with a bar chart

# Compute the range of each feature (Maximum Value - Minimum Value)
feature_ranges = X.max() - X.min()

feature_names = feature_ranges.index.tolist()
range_values = feature_ranges.values

# Create the horizontal bar chart
plt.figure(figsize=(10, 8))
plt.barh(feature_names, range_values, color='skyblue', edgecolor='navy', alpha=0.6)

plt.title('Range of values for each feature')
plt.xlabel('Range (Maximum - Minimum)')
plt.ylabel('Features')

# Invert the Y axis to read the features from top to bottom
plt.gca().invert_yaxis()

# Add a grid to make the numbers easier to read
plt.grid(axis='x', linestyle='--', alpha=0.7)

# Optimise the spacing so that labels are not cut off
plt.tight_layout()

plt.show()


<details>
<summary><b>⚠️ The problem: why can large numbers fool the AI?</b></summary>

### Variables on different scales

Our dataset contains a mix of data:
* **Continuous variables:** large numbers (e.g. Platelets up to 850,000).
* **Boolean variables:** only 0 or 1 (e.g. Smoker Yes/No).

**Why is this a problem?**
The algorithm is "fooled" by large numbers: it would think that platelets are thousands of times more important than smoking just because their values are higher. This would completely unbalance the distance computation in clustering.
</details>

---


## The solution: standardisation (Scaling)

To bring every feature onto the same scale, we apply this mathematical formula to each value:

$$z = \frac{x - \mu}{\sigma}$$

**Where:**
* **$x$**: is the original value (e.g. 70 years).
* **$\mu$ (mu)**: is the **mean** of that column.
* **$\sigma$ (sigma)**: is the **standard deviation** (it tells how varied or "spread out" the data are).
* **$z$**: is the final "scaled" value.

**What have we achieved?** After this transformation, the value **0** represents exactly the mean. A positive value means "above the mean", a negative one "below the mean". Now the computer can compare age with platelet levels without getting confused!

In [ ]:
# Scale the features before clustering using StandardScaler
from sklearn.preprocessing import StandardScaler

# Retrieve the original data (we keep a copy to be safe)
heart_failure_clinical_records = fetch_ucirepo(id=519)
X_original = heart_failure_clinical_records.data.features
y = heart_failure_clinical_records.data.targets

# Initialise the scaler (the tool that applies the Z-score formula)
scaler = StandardScaler()

# Apply scaling to the data: now every column will have the same weight
X_scaled = scaler.fit_transform(X_original)

print("Scaling successfully applied to the data")

In [ ]:
import pandas as pd

# Turn the scaled data into a readable table with the original column names
X_scaled_df = pd.DataFrame(X_scaled, columns=X_original.columns)

# Print the first 5 rows to see the difference
X_scaled_df.head()

In [ ]:
# Elbow Method on SCALED data

# List to store the WCSS (Inertia) values
wcss = []
k_range = range(1, 11)

# Compute the inertia for each k tested (from 1 to 10)
for k in k_range:
    # Use the scaled data (X_scaled) for a correct computation of distances
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Create the plot
plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss, marker='o', color='blue', linestyle='--')
plt.title('Elbow Method (on scaled data)')
plt.xlabel('Number of groups (k)')
plt.ylabel('Inertia (WCSS)')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
# Silhouette analysis on SCALED data to validate the number of groups

silhouette_scores = []
k_range = range(2, 11)

# Compute the silhouette score for each k (number of groups)
for k in k_range:
    # Use the scaled data for a correct evaluation of the separation
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    cluster_labels = kmeans.fit_predict(X_scaled)

    # The score measures how compact and well separated the clusters are
    score = silhouette_score(X_scaled, cluster_labels)
    silhouette_scores.append(score)
    # print(f'For k = {k}, the Silhouette score is: {score:.4f}')

# Create the Silhouette plot
plt.figure(figsize=(10, 6))
plt.plot(k_range, silhouette_scores, marker='o', color='green', linestyle='--')
plt.title('Silhouette Score (on scaled data)')
plt.xlabel('Number of groups (k)')
plt.ylabel('Silhouette Score')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
# Apply k-means with the optimal k on SCALED data and 2D plot with PCA
from sklearn.decomposition import PCA

# Choose 2 as the optimal k (based on the previous Elbow and Silhouette plots)
OPTIMAL_K = 2

# Dimensionality reduction (PCA) on SCALED data
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Apply K-Means on SCALED data
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init='auto')
y_kmeans = kmeans.fit_predict(X_scaled)

# Visualise the clusters
plt.figure(figsize=(10, 6))

# Draw the points (patients) coloured by their assigned group
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_kmeans, s=50, cmap='RdYlBu', alpha=0.7)

# Place the centroids (the group centres) in PCA space
centers_scaled = kmeans.cluster_centers_
centers_pca = pca.transform(centers_scaled)

# Draw the Centroids as big red 'X' markers
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', s=250, marker='X', label='Centroids')

plt.title(f'K-Means Clustering (k={OPTIMAL_K}) visualised with PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.grid(True)
plt.show()

### Group profiles: what sets them apart?

Now that we have split the patients into two groups, we need to understand "who" they are. We use a **Heatmap** to visualise the average profile of each cluster.

Since we are using **scaled** data (Z-score):
* **Positive values (red):** indicate that the feature is **above** the global mean.
* **Negative values (blue):** indicate that the feature is **below** the global mean.
* **Values close to 0 (white/grey):** indicate values in line with the mean of all patients.

This lets us see at a glance which clinical factors define one group compared to the other.

In [ ]:
# Heatmap of the cluster profiles based on the scaled data
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Create a DataFrame with the scaled data and add the cluster column
X_analysis_scaled = pd.DataFrame(X_scaled, columns=X_original.columns)
X_analysis_scaled['cluster'] = y_kmeans

# Compute the mean of each feature for each cluster
# These means represent the Z-score profiles (distance from the global mean)
scaled_means = X_analysis_scaled.groupby('cluster').mean()

# Generate the Heatmap
plt.figure(figsize=(10, 10))
sns.heatmap(
    scaled_means.T, # Transpose the table to have the variables on the rows
    annot=True,     # Show the numbers inside the cells
    cmap='coolwarm', # Blue for low values, red for high values
    fmt=".2f",
    center=0,       # Zero (the global mean) maps to white
    cbar_kws={'label': 'Cluster Mean (Z-score relative to the global mean)'}
)

plt.title('Heatmap of Cluster Profiles (Scaled Data)')
plt.ylabel('Clinical Variables')
plt.xlabel('Cluster')
plt.xticks(rotation=0)
plt.show()

<details>
<summary><b>🕵️‍♂️ Interpreting the results: who are the patients?</b></summary>

From the analysis of the previous Heatmap, we can draw a clear profile of the two groups created by the clustering:

* **Cluster 0 (the male profile):** it is made up of **men**, both smokers and non-smokers. In the heatmap we see positive values (red) for the variables `sex` and `smoking`: compared to the global mean, there are more men and more smokers here.
* **Cluster 1 (the female profile):** it is made up almost only of **non-smoking women**. Here the values are negative (blue): almost no smokers and almost no men (in the dataset `sex = 1` means men).
</details>


---


<details>
<summary><b>🤔 Why does the AI choose exactly these variables to create the groups?</b></summary>

#### Technical remark: the weight of binary variables
We notice that the **binary variables** (those with only two states: 0 or 1, such as smoking, sex, diabetes) seem to "drive" the formation of the groups.

This often happens in clustering: since these variables create a sharp separation (you are either in or out), the algorithm finds them very "convenient" for splitting the data into distinct blocks. In this case, the AI decided that sex and smoking habits were the strongest traits for distinguishing patients in the dataset.
</details>

---

### Putting the results together: back to the real data

Even though we used the "scaled" data to make the algorithm work properly, we now want to analyse the patients using their original values.

We add a new column called `cluster` to our original dataset. This way, we can easily see for each patient whether the AI placed them in **Group 0** or in **Group 1**.

In [ ]:
# Add the cluster labels to the original dataframe X
# This way we can study the real clinical values of each group
X['cluster'] = y_kmeans

# Print a confirmation message and the first rows of the updated table
print("Cluster labels successfully added to the original dataset.")
X.head()

### Statistical analysis: what do the numbers say?

To really understand who belongs to the two groups, we analyse the average statistics.

**Note for the class:**
1.  **Continuous Variables:** we look at the mean and the median to understand the "typical patient" of the group. The values are Z-scores, i.e. they show how far above (positive) or below (negative) the global mean the group is.
2.  **Binary Variables (0/1):** here is a trick! The **mean** represents the percentage. If we read `smoking: mean 0.85`, it means that **85%** of the patients in that group smoke.

In [ ]:
# Detailed statistics for each cluster
# For continuous variables we use the scaled data to see where each group stands with respect to the global mean (0)
groups = X_analysis_scaled.groupby('cluster')

# Distinguish between continuous and binary variables for a clearer reading
for name, group in groups:
    print(f"\n" + "="*30)
    print(f" CLUSTER {name} PROFILE ")
    print("="*30)

    # Continuous variables: show mean, standard deviation and median
    print("\n[Continuous Variables - Z-score Profile]")
    print(group.agg({
            'age': ['mean', 'std', 'median'],
            'creatinine_phosphokinase': ['mean', 'std', 'median'],
            'ejection_fraction': ['mean', 'std', 'median'],
            'platelets': ['mean', 'std', 'median'],
            'serum_creatinine': ['mean', 'std', 'median'],
            'serum_sodium': ['mean', 'std', 'median'],
            'time': ['mean', 'std', 'median'],
        }))

    # Binary variables: we use the original data (0/1), so the mean gives the proportion of "Yes" (1) in the group
    print("\n[Binary Variables - Proportions and Count]")
    print(X[X['cluster'] == name].agg({
            'anaemia': ['mean', 'count'],
            'diabetes': ['mean', 'count'],
            'high_blood_pressure': ['mean', 'count'],
            'sex': ['mean', 'count'],
            'smoking': ['mean', 'count'],
        }))

In [ ]:
# Separate the variable types for visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# Continuous variables (to be visualised with Boxplots)
continuous_vars = [
    'age', 'creatinine_phosphokinase', 'ejection_fraction',
    'platelets', 'serum_creatinine', 'serum_sodium', 'time'
]

# Binary variables (to be visualised with proportion bar charts)
binary_vars = [
    'anaemia', 'diabetes', 'high_blood_pressure', 'sex', 'smoking'
]

### How to read a boxplot

The **boxplot** is one of the most powerful tools to see how data are distributed. Here is how to interpret each of its parts:

1. **The box:** represents the core of the data, i.e. the **central 50%** of the sample.
   * The lower edge is the **25th percentile** (Q1): 25% of patients have a value below it.
   * The upper edge is the **75th percentile** (Q3): 75% of patients have a value below it.
2. **The central line:** represents the **median** (50th percentile). It is the value that splits the patients exactly in half. If the line is not in the middle of the box, the data are skewed (asymmetric).
3. **The whiskers:** the lines extending above and below the box. They show the range where the "normal" data lie (usually up to 1.5 times the height of the box).
4. **The dots (outliers):** if you see points beyond the whiskers, these are the so-called outliers. They are patients with very rare or anomalous values compared to the rest of the group.
5. **Notch:** helps to compare medians: if the notches of two groups do not overlap, the difference between the medians of the two groups is very likely to be statistically significant.

In [ ]:
# Boxplots for the continuous variables
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(16, 12))

for i, col in enumerate(continuous_vars, 1):
    plt.subplot(3, 3, i)

    sns.boxplot(x='cluster', y=col, data=X, notch=True, palette='Set2', hue='cluster', legend=False, boxprops={'alpha': 0.7})

    plt.title(f'Distribution of: {col.replace("_", " ").capitalize()}')
    plt.xlabel('Group (Cluster)')
    plt.ylabel('Real Value')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

### Analysing binary variables (Yes/No)

For variables such as **smoking**, **diabetes** or **sex**, we cannot use means in the traditional sense. Instead, we use a **count plot**.

In these plots:
* **The X axis** splits the patients into the two Clusters (0 and 1).
* **The colours (hue)** indicate the presence (1) or absence (0) of a feature.
* **The height of the bars** tells us how many patients belong to that category.

This is the moment of truth: we will clearly see whether a group is made up almost entirely of men or whether smoking is concentrated in only one cluster.

In [ ]:
# Count Plots for the binary variables arranged in a grid
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 10))

for i, col in enumerate(binary_vars, 1):
    plt.subplot(2, 3, i)

    # Create the bar chart by counting the occurrences
    ax = sns.countplot(data=X, x='cluster', hue=col, palette='viridis', alpha=0.7)

    plt.title(f'Distribution of: {col.replace("_", " ").capitalize()}')
    plt.xlabel('Group (Cluster)')
    plt.ylabel('Number of Patients')

    if col == "sex":
        labels = ['Women (0)', 'Men (1)']
    else:
        labels = ['No (0)', 'Yes (1)']

    # Retrieve the legend and change its labels
    plt.legend(title=col.replace("_", " ").capitalize(), labels=labels)

# Clean up the layout
plt.tight_layout()
plt.show()

<details>
<summary><b>👀 What do we observe when analysing the groups?</b></summary>

* **Cluster 0:** exclusively men, about half smokers and half non-smokers.
* **Cluster 1:** almost exclusively non-smoking women.
</details>

<details>
<summary><b>🧩 Why does this happen? (The hidden limitation)</b></summary>

In this specific database (collected in a specific clinical setting), the category of **female smokers** is practically absent.

As a consequence, the clustering algorithm has "learned" that smoking is an almost exclusively male trait. AI has no "common sense": it does not know that in the real world there are many women who smoke; it only sees what the data show it.
</details>

In [ ]:
# Filter the dataset to find only the women who smoke
female_smokers = X[(X['sex'] == 0) & (X['smoking'] == 1)]

# Count how many they are compared to the total number of women
total_women = len(X[X['sex'] == 0])
num_female_smokers = len(female_smokers)

print(f"Total patients in the dataset: {len(X)}")
print(f"Total women: {total_women}")
print(f"Of whom smokers: {num_female_smokers}")

print(f"\nPercentage of smokers among women: {(num_female_smokers/total_women)*100:.2f}%")


### The key lesson: "garbage in, garbage out"
Artificial intelligence has no "common sense": it does not know that in the real world there are many women who smoke. It simply finds patterns in the data we provide.
If the data are unbalanced or reflect cultural/geographical biases, the groups created by the AI will be distorted. **The model does not describe the absolute truth, only a mirror of the data it was given.**

---

### Which variables "drive" the groups?

After looking at the plots, we ask mathematics to confirm which features really guided the AI. We use statistical tests to find out which variables "drove" the split of the patients and which ones are just due to chance.

In [ ]:
import pandas as pd
from scipy import stats

# Split the original data into the two identified clusters
cluster_0 = X[X['cluster'] == 0]
cluster_1 = X[X['cluster'] == 1]

alpha = 0.05 # Standard significance threshold
results_list = []

# 1. Kruskal-Wallis test for continuous variables
for var in continuous_vars:
    stat_kw, p_value_kw = stats.kruskal(cluster_0[var], cluster_1[var])
    results_list.append({
        'Feature': var,
        'Test': 'Kruskal-Wallis',
        'P-Value': p_value_kw,
        'Significant': 'Yes' if p_value_kw < alpha else 'No'
    })

# 2. Chi-squared test for binary variables
for var in binary_vars:
    contingency_table = pd.crosstab(X['cluster'], X[var])
    # Check that the table is suitable for the test (2x2)
    if contingency_table.shape == (2, 2):
        chi2, p_value_chi2, _, _ = stats.chi2_contingency(contingency_table)
    else:
        p_value_chi2 = 1.0

    results_list.append({
        'Feature': var,
        'Test': 'Chi-squared',
        'P-Value': p_value_chi2,
        'Significant': 'Yes' if p_value_chi2 < alpha else 'No'
    })

# Create the summary DataFrame
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(by='P-Value')

results_df.drop(columns=['P-Value', 'Test'], inplace=True)
results_df

<details>
<summary><b>🔍 Summing up: what really guided the algorithm?</b></summary>

### What did we discover?
The algorithm "saw" **sex** and **smoking** as the main coordinates to find its way.

This happens because binary variables (Yes/No, Man/Woman) create very sharp boundaries that K-Means captures easily, while the continuous medical variables (such as platelets) were only used to further refine the groups. In practice, the AI chose the "easy way" to split the patients, even though it was not the most useful one from a medical point of view.
</details>

---

## Clinical validation: do the clusters reflect the risk of death?

After splitting the patients by sex and smoking (the dominant factors in this dataset), we must ask ourselves: **does this split have any medical value?** We cross our clusters with the `death_event` column (the death event) to see whether one of the two groups has a higher mortality rate.

In [ ]:
# Analysis of the mortality rate (death_event) for each cluster

# Add the real death variable to dataframe X (using the targets y)
# Note: y contains the 'death_event' column
X['death_event'] = y.values

# Compute the exact percentages for the discussion
mortality_rates = X.groupby('cluster')['death_event'].mean() * 100

print(f"Mortality rate Cluster 0: {mortality_rates[0]:.2f}%")
print(f"Mortality rate Cluster 1: {mortality_rates[1]:.2f}%")

# Remove the 'death_event' column from X to avoid confusion in later analyses
X.drop(columns=['death_event'], inplace=True)

---

## The final verdict

We asked the AI to split the patients and it did so perfectly. But now let's face the facts...

<details>
<summary><b>📊 1. Let's look at the numbers: what is the mortality rate in the two groups?</b></summary>

* **Cluster 0 rate (Men):** ~32.3%
* **Cluster 1 rate (Non-smoking women):** ~31.8%

**Surprise:** the two groups have practically the same risk of death.
</details>

<br>

<details>
<summary><b>🎓 2. What did we learn from this unexpected result?</b></summary>

1.  **Structure ≠ Risk:** the algorithm found a "structure" (it split men and women), but this split **has no clinical relevance** for survival.
2.  **"Noise" Variables:** sex and smoking were chosen by the AI only because they were easy to split (0 or 1), but they are not the real culprits of mortality.
3.  **The importance of medical data:** factors such as age, creatinine or ejection fraction matter much more, even though they are harder to group by eye.
</details>

<br>

<details>
<summary><b>🏁 3. The moral of the story (The Data Scientist's lesson)</b></summary>

**Not all patterns found by Artificial Intelligence are useful.** In this experiment we found a **sociological** pattern (men vs non-smoking women), but we failed to find a **medical** pattern (who is really more at risk).

*Remember: AI sees the differences, but it is up to us humans to decide which differences really matter.*
</details>

---

### 1. Supervised Learning
We move from clustering to **Classification**. In this phase the AI is **supervised**: we provide it with the patients' data together with the correct answer (**death_event**). The model learns from past cases to predict the risk of future patients.

![Supervised vs Unsupervised Learning](https://upload.wikimedia.org/wikipedia/commons/4/4d/Supervised_and_unsupervised_learning.png)

### 2. Decision Tree and Random Forest
* **Decision Tree:** a flowchart that asks questions to the data (e.g. "Age > 70?") to reach a conclusion.

* **Random Forest:** a "forest" made of many independent trees (here 100). Each tree casts a vote and the final decision is taken by **majority**. It is much more robust and accurate than a single tree.

<img src="https://miro.medium.com/v2/resize:fit:1400/1*b3kM3WFJB6b92ZHp_J871Q.jpeg" style="width:38%;">

### 3. Goal: Feature Importance
We use the Random Forest to compute the **Feature Importance**. This will let us understand whether, for the AI, the clinical parameters (heart and kidneys) or the social factors (sex and smoking) that emerged earlier with clustering are more decisive.

> **Watch out for the `time` variable:** it is the length of the follow-up, i.e. for how many days the patient was monitored. Patients who die early necessarily have a short follow-up: this variable therefore already "knows" the outcome, and this is why it will turn out to be the most important one. However, it is not a clinical factor, so when reading the results we mainly look at the other variables.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# Create a simple decision tree to understand which features are most important to predict death

# We use X_original (original data, without the 'cluster' column) and not X_scaled, so the questions will be readable
tree_simple = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_simple.fit(X_original, y)

# Visualise the tree
plt.figure(figsize=(20, 10))
nodes = plot_tree(tree_simple,
                  feature_names=X_original.columns,
                  class_names=['Survived', 'Deceased'],
                  filled=True,
                  rounded=True,
                  fontsize=12,
                  impurity=False)

# Loop to clean up the text
for n in nodes:
    text = n.get_text().split('\n')
    # Cleanup: if there is a condition we keep the first and the last line.
    # If it is a leaf, we only keep the final result.
    if "<=" in text[0]:
        n.set_text(f"{text[0]}\n{text[-1]}")
    else:
        n.set_text(text[-1])

plt.title("The Decision Tree: the AI's 'Reasoning'", fontsize=20, pad=20)
plt.show()

In [ ]:
# Classification task to predict death ('death_event') with a Random Forest
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Data preparation: split the dataset into Training (80%) and Test (20%) sets
# We use 'stratify=y' to make sure the proportion of deaths is the same in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Model Initialisation and Training
# The Random Forest builds a "forest" of decision trees for a more robust prediction
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train.values.ravel()) # .values.ravel() avoids format errors with Pandas

# 3. Prediction on the Test data (data the model has never seen)
y_pred_rf = rf_clf.predict(X_test)

# 4. Performance Evaluation
print("\n--- Random Forest Results ---")
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Model Accuracy: {accuracy_rf:.2f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

# 5. Confusion Matrix: how many did we get right and where did we go wrong?
plt.figure(figsize=(8, 6))
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Survived', 'Deceased'], yticklabels=['Survived', 'Deceased'])
plt.title('Confusion Matrix - Random Forest')
plt.ylabel('True Value')
plt.xlabel('Predicted Value')
plt.show()

# 6. Feature Importance analysis
# Let's see which factors weighed the most in the model's decision
feature_importances = pd.DataFrame(
    rf_clf.feature_importances_,
    index = X_original.columns,
    columns=['importance']
).sort_values('importance', ascending=False)

print("\nFeature importance according to the Random Forest:")
print(feature_importances)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 1. Compute the importance for PREDICTING THE CLUSTERS
rf_clusters = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clusters.fit(X_scaled, y_kmeans)
imp_clusters = pd.Series(rf_clusters.feature_importances_, index=X_original.columns).sort_values(ascending=False)

# 2. Compute the importance for PREDICTING DEATH
rf_death = RandomForestClassifier(n_estimators=100, random_state=42)
rf_death.fit(X_scaled, y.values.ravel())
imp_death = pd.Series(rf_death.feature_importances_, index=X_original.columns).sort_values(ascending=False)

# 3. Comparison plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

# Plot Cluster Importance
sns.barplot(x=imp_clusters.values, y=imp_clusters.index, hue=imp_clusters.index, ax=ax1, palette="Blues_r", legend=False)
ax1.set_title("What defines the CLUSTERS?", fontsize=13, fontweight='bold')
ax1.set_xlabel('Importance')
ax1.set_ylabel('')

# Plot Death Importance
colors = ['red' if (idx == 'sex' or idx == 'smoking') else 'gray' for idx in imp_death.index]
sns.barplot(x=imp_death.values, y=imp_death.index, hue=imp_death.index, ax=ax2, palette=colors, legend=False)
ax2.set_title("What defines MORTALITY?", fontsize=13, fontweight='bold')
ax2.set_xlabel('Importance')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

## 🏁 What have we learned?

This final comparison reveals the algorithm's "trick" and teaches us a fundamental lesson in medicine and data science.

<details>
<summary><b>🤔 1. Was the clustering "distracted"?</b></summary>

Yes! The clustering split the patients by **sex and smoking** (the blue bars we saw on the left) only because they were the easiest differences to detect mathematically.
It created "social" groups, but it told us nothing about the real heart health of the patients.
</details>

<br>

<details>
<summary><b>🩺 2. Did the Random Forest find the truth?</b></summary>

Exactly. When we gave the model the goal of predicting mortality, the sex and smoking variables "collapsed" (red bars on the right).
The AI learned that, to save a life, we must ignore sex and look at the vital data: **the heart and the kidneys** (leaving aside `time` which, as we saw, is not a clinical factor).
</details>

<br>

<details>
<summary><b>💡 Conclusion: the power of Visualisation</b></summary>

A clear visualisation of the data is not just for making "nice plots". It allows us to understand when a model is following a **data bias** rather than real medical usefulness.

**A good Data Scientist is one who knows how to question the machine's results!**
</details>

---